# 🚘 VehicleAI — How the Machine Learning System Works

**Developer portfolio / reproducible Jupyter walkthrough** — detection, tracking, directional counting, evaluation and optional custom fine-tuning.

**Important honesty note:** This application starts with an **Ultralytics pretrained YOLO26n detector**, trained by the model provider. The author developed the Python processing pipeline, ByteTrack integration, counting algorithm and web dashboard. The project does **not** claim to have trained YOLO26 from scratch. The *optional* custom-training chapter describes how to extend it with your own labeled dataset. No evaluation figures are invented.

> Requirements: Python 3.11, `pip install -r requirements.txt`, plus `pip install jupyterlab matplotlib` (if not installed). The first YOLO model load may download weights from the internet.
> Use footage you own or have permission to process. Save a short clip as `samples/traffic.mp4` from the project root.

## 1 · Architecture and learning objectives

**Python development pipeline:** Video → OpenCV frame → pretrained YOLO26 → ByteTrack ID → counting-line crossing → aggregate counts.

**V8 Cloudflare browser deployment:** Browser `<video>` / webcam → browser ONNX YOLO26 on WebGPU/CPU → lightweight sampled-frame tracker in JavaScript → crossing counts → optional summary-only Cloudflare D1 storage.

These implementations share the same *concept* but use different tracking engines: ByteTrack in this notebook, browser proximity/IoU matching in V8. They will not necessarily produce identical totals; V8 cannot claim every-frame tracking or identical accuracy. The notebook demonstrates real ML inference, but does not train YOLO26 from scratch.


## 2 · Install requirements and choose a video

Run from Anaconda Prompt in the **cloudflare-v8** directory:

```bat
conda activate vehicle-ai
python -m pip install -r requirements-notebook.txt
python -m jupyter lab
```

Open this notebook in JupyterLab. Put your own licensed video in `samples/traffic.mp4`. These local Python examples are for developer education; Cloudflare Pages executes the browser ONNX model, not this notebook.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
VIDEO_PATH = PROJECT_ROOT / "samples" / "traffic.mp4"
MODEL_NAME = "yolo26n.pt"       # pretrained YOLO26 nano; or set path to trained best.pt
VEHICLE_CLASS_IDS = [1, 2, 3, 5, 7]
LINE_RATIO = 0.55               # adjust for camera angle; 0.15 to 0.85
print("Project root:", PROJECT_ROOT)
print("Video exists:", VIDEO_PATH.is_file())
print("If False, copy a licensed video to samples/traffic.mp4 before processing")

## 3 · The neural network: load YOLO and inspect labels

YOLO26 predicts bounding-box coordinates, class scores and confidence values from an input frame. We are **using existing pretrained weights**, then restricting results to the vehicle-related COCO categories. Changing the model weights, data or architecture is not necessary to run this demo.

Useful primary references: [Ultralytics model guides](https://docs.ultralytics.com/models/) and [tracking guide](https://docs.ultralytics.com/modes/track/).

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL_NAME)  # downloads pretrained weights once if missing
print('Selected COCO classes:')
for class_id in VEHICLE_CLASS_IDS:
    print(f'{class_id}: {model.names[class_id]}')

## 4 · Read a single real frame

Start with one frame to understand what the detector receives. OpenCV reads frames in **BGR** order; matplotlib displays **RGB**.

In [ ]:
import cv2
import matplotlib.pyplot as plt

if not VIDEO_PATH.is_file():
    raise FileNotFoundError(f'Add your own traffic clip to: {VIDEO_PATH}')
cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), 'OpenCV could not open this video — try MP4/H.264.'
ok, frame = cap.read()
cap.release()
assert ok, 'Could not read the first frame.'
print('H x W x C:', frame.shape)
plt.figure(figsize=(13, 7))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.axis('off'); plt.title('Input frame (no predictions yet)');

## 5 · Run a pretrained detector on the sample frame

The **first stage** produces only bounding boxes. It does not yet count vehicles. In this cell `classes=` filters COCO categories before plotting results.

In [ ]:
results = model.predict(frame, classes=VEHICLE_CLASS_IDS, conf=0.28,
                        imgsz=640, verbose=False)
result = results[0]
print('Detected objects:', len(result.boxes) if result.boxes is not None else 0)
if result.boxes is not None:
    for cls_id, confidence in zip(result.boxes.cls.int().cpu().tolist(), result.boxes.conf.cpu().tolist()):
        print(model.names[cls_id], f'{confidence:.2%}')
plt.figure(figsize=(13, 7))
plt.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
plt.axis('off'); plt.title('YOLO26n pretrained object detection (single frame)');

## 6 · ByteTrack and line crossing

ByteTrack associates boxes with track IDs over adjacent frames. `persist=True` means **these are successive frames of the same video**, not unrelated pictures. The custom `TrafficCounter` only increments after a given ID moves from one **stable side** of the line to the other. A small tolerance/dead zone reduces noisy side changes.

**Failure cases to recognize:** detector misses an object, tracker changes its ID, the object is occluded, or movement is parallel to the line. Counted events therefore need validation against manual ground truth.

In [ ]:
from collections import defaultdict

class TrafficCounter:
    """Count a tracked vehicle only once when its center crosses a horizontal line."""
    def __init__(self, line_ratio=0.55):
        self.line_ratio=line_ratio
        self.previous_side={}
        self.counted=set()
        self.counts=defaultdict(lambda: {'up':0, 'down':0})

    def observe(self, track_id, name, center_y, frame_height):
        side = 'above' if center_y < frame_height * self.line_ratio else 'below'
        previous=self.previous_side.get(track_id)
        self.previous_side[track_id]=side
        if previous is None or previous==side or track_id in self.counted:
            return False
        direction = 'up' if side=='above' else 'down'
        self.counts[name][direction]+=1
        self.counted.add(track_id)
        return True

    def total(self):
        return sum(v['up']+v['down'] for v in self.counts.values())

counter_example = TrafficCounter(line_ratio=0.55)
print('Before crossing:', counter_example.total())
print('First observation:', counter_example.observe(track_id=7, name='car', center_y=75, frame_height=100))
print('Crossing upward:', counter_example.observe(track_id=7, name='car', center_y=35, frame_height=100))
print('Repeated crossing:', counter_example.observe(track_id=7, name='car', center_y=75, frame_height=100))
print('Final number of crossings:', counter_example.total())
print('Counts:', dict(counter_example.counts))


## 7 · Process multiple frames with real tracking

This cell runs the detector + tracker + counting pipeline on up to **90 frames**. Use a short video/CPU-friendly value first; increase when you want to analyze more. Always use a **fresh model** for each independent video so track IDs do not leak across streams.

In [ ]:
from ultralytics import YOLO
import cv2

fresh_model = YOLO(MODEL_NAME)
counter = TrafficCounter(line_ratio=LINE_RATIO)
video = cv2.VideoCapture(str(VIDEO_PATH))
last_annotated = None
MAX_FRAMES = 90  # increase for a longer real video; CPU processing can be slow
frames_processed = 0
while frames_processed < MAX_FRAMES:
    success, current = video.read()
    if not success:
        break
    tracked = fresh_model.track(current, persist=True, tracker='bytetrack.yaml',
                                classes=VEHICLE_CLASS_IDS, conf=0.28, verbose=False)[0]
    last_annotated = tracked.plot()
    if tracked.boxes is not None and tracked.boxes.id is not None:
        for xyxy, cls, track_id in zip(tracked.boxes.xyxy.cpu().numpy(),
                                       tracked.boxes.cls.int().cpu().tolist(),
                                       tracked.boxes.id.int().cpu().tolist()):
            center_y = (xyxy[1]+xyxy[3])/2
            counter.observe(track_id, fresh_model.names[cls], center_y, current.shape[0])
    y = int(current.shape[0] * LINE_RATIO)
    cv2.line(last_annotated, (0,y), (current.shape[1],y), (0,255,255), 3)
    cv2.putText(last_annotated, f'Crossings: {counter.total()}', (15,40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255,255,255), 2)
    frames_processed += 1
video.release()
print('Frames processed:', frames_processed)
print('Counted crossings:', counter.total())
print('By class / direction:', dict(counter.counts))
if last_annotated is not None:
    plt.figure(figsize=(13,7))
    plt.imshow(cv2.cvtColor(last_annotated, cv2.COLOR_BGR2RGB))
    plt.axis('off'); plt.title('Tracked vehicles & counting line')


## 8 · Evaluate honestly against manual annotations

The following is a template, **not a fabricated evaluation**. Replace `ground_truth` with manually counted crossing events from the **same segment of video** as the code above. MAE measures how different the predicted counts are from ground-truth counts. If you have not counted the video manually, the code leaves metrics unreported.

In [ ]:
ground_truth = None  # EXAMPLE after annotation: {'car': 3, 'motorcycle': 1, 'bus': 0, 'truck': 0, 'bicycle': 0}
predicted = {name: counts['up'] + counts['down'] for name, counts in counter.counts.items()}
if ground_truth is None:
    print('No ground truth provided; MAE and count accuracy are NOT reported.')
else:
    categories = sorted(set(ground_truth) | set(predicted))
    errors = {cat: abs(predicted.get(cat,0) - ground_truth.get(cat,0)) for cat in categories}
    mae = sum(errors.values()) / len(errors)
    print('Predicted:', predicted)
    print('Ground truth:', ground_truth)
    print('Absolute errors:', errors)
    print('MAE:', mae)

## 9 · Optional: how you would train a custom traffic model

If you need a **Sri Lankan three-wheeler** class, you cannot expect COCO weights to provide it automatically. The typical workflow is:

1. Collect licensed images across roads, cameras, lighting and weather; split by scene, not random near-duplicate frames.
2. Annotate bounding boxes and classes such as `car`, `motorcycle`, `bus`, `truck`, `three_wheeler`, `bicycle`.
3. Create train/val/test partitions; keep test images unseen during training.
4. Convert labels to YOLO format and create a `dataset.yaml` mapping class IDs to names.
5. Fine-tune a suitable pretrained model, inspect mAP50–95 and per-class errors, and export your `best.pt`.
6. Update `MODEL_NAME` in `.env` and **also update the class-ID mapping in `app/vision.py`** to match your new dataset.

The example below is a **template only**; it does not run without a dataset. Training from scratch is neither necessary nor claimed for this demo.

In [ ]:
# Example dataset.yaml (create this file yourself alongside your labeled images):
# path: C:/datasets/sri_lanka_traffic
# train: images/train
# val: images/val
# test: images/test
# names:
#   0: car
#   1: motorcycle
#   2: bus
#   3: truck
#   4: three_wheeler
#   5: bicycle

# Fine-tune ONLY after collecting and validating real annotations:
# from ultralytics import YOLO
# custom = YOLO('yolo26n.pt')
# custom.train(data='dataset.yaml', epochs=40, imgsz=640, batch=8, project='runs/custom_traffic')
# metrics = custom.val(data='dataset.yaml', split='test')
# print(metrics)
print('This is an optional documented training workflow; no custom training was performed here.')

## 10 · Connecting the ML code to the web application

The frontend shows a local video preview using `URL.createObjectURL(file)` without waiting for inference. The Python API receives the uploaded file at `POST /api/videos` and analyzes it in a background task. On completion, the browser plays the real annotated output from `GET /api/videos/{id}/play` and can download the MP4/CSV. The webcam calls `POST /api/live/start` and sends JPEG frames to `POST /api/live/{id}/frame`.

The welcome email and Request Demo forms use separate APIs and optional SMTP credentials from `.env`; the AI model does not need email or Google Sign-in to run.

**What this project demonstrates:** deep-learning inference, tracking, engineered event-counting logic, frontend/back-end integration, a repeatable notebook, and limitations disclosed clearly. The model's weights come from Ultralytics. Model/package license and footage permissions matter for public/commercial use.

### Suggested next experiments
- Evaluate on *your own labeled traffic clips*, saving confusion cases and count errors.
- Compare CPU vs GPU inference speed, and YOLO nano vs a larger model.
- Study IDs around occlusions and adjust ByteTrack configuration.
- For a custom tuk-tuk class, label data carefully and fine-tune only after creating an evaluation split.

References: [Ultralytics tracking docs](https://docs.ultralytics.com/modes/track/), [Ultralytics models](https://docs.ultralytics.com/models/), [OpenCV](https://docs.opencv.org/).

## V6 · Why we process every uploaded frame

A web browser can play 25–30 frames each second while a CPU may process fewer than 5 YOLO frames per second. Sampling the currently playing picture every 0.5 seconds can cause ByteTrack to lose IDs, so vehicles crossing the line may not be counted.

**V6 uploaded-video pipeline**: read each frame sequentially with OpenCV, run YOLO + ByteTrack on each one, count crossings, save an annotated video, and collect crossing timestamps. The website only receives a small JPG *preview* every 12 analyzed frames to avoid flooding the connection. Do not confuse preview-frame sampling with analysis-frame sampling: the **analysis processes all frames**.

**Live webcam / direct URL**: resource-limited browser sampling is still used and may miss fast-moving vehicles. Offline processing is preferred when accuracy matters.


In [ ]:
# Inspect frame rate and estimate number of frames to analyze
import cv2
from pathlib import Path

# Set VIDEO_PATH earlier in this notebook before running.
if VIDEO_PATH.is_file():
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    print(f"Full scan: {total_frames:,} frames at {fps:.1f} FPS")
    print(f"Video duration: {total_frames/max(fps,1):.1f} seconds")
else:
    print("Add a sample MP4 in the samples folder and configure VIDEO_PATH first.")


## V6 · Playback-time count reconstruction

For each crossing event the server stores `{time, vehicle_type, direction, track_id}`. The website can replay the annotated MP4 and reconstruct the exact count at any timestamp. This is more useful than pretending the CPU's live processing speed always matches the native video player.

A vehicle must cross the chosen horizontal line with its tracked center; a vehicle in the picture does not increment *line crossings*. Poor camera placement, occlusion, poor lighting, wrong line position and tracking ID changes can still cause undercounting or double counting. A benchmark on annotated ground-truth clips is required before claiming accuracy.


In [ ]:
# Self-contained demonstration of time-aligned counting (no YOLO inference required)
events = [
    {"time": 0.8, "vehicle_type": "car", "direction": "down", "track_id": 17},
    {"time": 1.5, "vehicle_type": "truck", "direction": "down", "track_id": 21},
    {"time": 2.7, "vehicle_type": "car", "direction": "up", "track_id": 19},
]

def counts_at(timestamp):
    return {"up": sum(x['direction'] == 'up' and x['time'] <= timestamp for x in events),
            "down": sum(x['direction'] == 'down' and x['time'] <= timestamp for x in events)}

for moment in [0.0, 1.0, 2.0, 3.0]:
    print(f"{moment:.1f}s -> {counts_at(moment)}")


## V6 · How to measure real-world accuracy

1. Collect video with appropriate use permission and minimize unnecessary personal data.
2. Annotate real line crossings and class labels, including occlusion cases. Label ground truth independently.
3. Compare predicted events to ground truth within a matching time tolerance. Report precision, recall, and errors by class and lighting condition.
4. Repeat on daytime/nighttime, camera motion, multiple lanes, and vehicle overlap scenarios.
5. For tuk-tuks, create a dedicated labeled Sri Lankan traffic dataset and fine-tune appropriately; standard COCO road-user classes do not contain a separate three-wheeler label.

This notebook demonstrates **pretrained model inference**; it is not evidence that a new neural network was trained from scratch.


## V8 · Export ONNX for real browser inference

The new free-hosted edition uses the official `@ultralytics/yolo` browser inference library. **The `.pt` weights cannot be uploaded to Cloudflare Pages as a Python service.** Export the pretrained model to `.onnx` instead. The Cloudflare static asset limit is 25 MiB per file. Check the size and the model license before deploying. The downloadable V8 source includes a Windows model-export script.


In [ ]:
from pathlib import Path
from ultralytics import YOLO
onnx_path = Path(YOLO(MODEL_NAME).export(format="onnx", imgsz=640))
print("ONNX model:", onnx_path, "bytes:", onnx_path.stat().st_size)
print("Model must be placed at cloudflare-v8/public/models/yolo26n.onnx")


## V8 · Browser tracking vs accurate video counting

Browser inference predicts boxes on actual observed frames. The JavaScript tracker matches boxes by proximity/IoU and counts a vehicle only when its tracked center crosses the line. Inference may run slower than the video, so the live mode samples frames. Sequential analysis **seeks to requested timestamps**; this is not guaranteed to decode every original frame. For rigorous benchmarks compare output against manually counted reference video and use the local Python/ByteTrack pipeline when needed.


## V9 · Why a van might be called a bus, or a truck might be called a bus

YOLO26n here is **pretrained on COCO, not trained by this VehicleAI project**. COCO has `car`, `motorcycle`, `bus`, `truck`, `bicycle` but **no separate `van` class**. So a true van may be given a `car`, `bus`, or `truck` label. Raising the confidence threshold may reduce weak false positives but **cannot guarantee the correct class**. Occlusion, perspective, video blur and small objects also cause confusion.

The browser V9 tracker reduces short-lived label flicker by using **class-independent spatial association and a short history of class votes**. It does not change what YOLO predicts. The V9 website allows **manual correction** (click a vehicle box, or a recorded crossing event, choose `van`, `truck`, etc.). A manual `van` is clearly marked as corrected, never presented as the model's prediction.

For truly automatic reliable `van` recognition, collect a **licensed, labeled Sri Lankan road dataset** containing separate van/car/bus/truck examples and **train a custom detection model**. Validate on separate video scenes before claiming accuracy.

## V9 · Investigate class confusion using real, labeled images

Prepare your own labeled images and a CSV of ground-truth categories. Below is a **template for manual ground truth**. It does not manufacture a confusion matrix or claim evaluation has been completed. Before interpreting any result, sample varied camera angles, vehicle sizes and day/night settings.

In [ ]:
# Fill each record with a real ground-truth vehicle crop path and the correct type.
# These are examples of expected structure ONLY; the list is deliberately empty.
validation_crops = [
    # {"image": "samples/crops/my_van_01.jpg", "truth": "van"},
    # {"image": "samples/crops/my_bus_01.jpg", "truth": "bus"},
]
from collections import Counter

if not validation_crops:
    print("Add your OWN labeled vehicle crop paths before generating a class-confusion report.")
else:
    ground_truth, predictions = [], []
    for item in validation_crops:
        prediction = model.predict(item["image"], conf=0.40, verbose=False)[0]
        if len(prediction.boxes) == 0:
            predicted = "missed"
        else:
            largest = max(prediction.boxes, key=lambda box: float(box.xywh[0,2]*box.xywh[0,3]))
            predicted = prediction.names[int(largest.cls.item())]
        ground_truth.append(item["truth"])
        predictions.append(predicted)
    for (truth, predicted), n in sorted(Counter(zip(ground_truth, predictions)).items()):
        print(f"actual={truth:12} predicted={predicted:12} count={n}")


## V9 · Optional custom training plan for van / bus / truck

1. Collect licensed footage that represents your deployment road camera; avoid retaining identifiable people unnecessarily.
2. Annotate each box with a class: `car`, `van`, `motorcycle`, `bus`, `truck`, `bicycle` (and `three_wheeler` if desired).
3. Split by **video/scene**, not random frames from one clip, into training, validation and test sets to avoid leakage.
4. Train a YOLO detection model on the new dataset; do not claim accuracy until you measure class-wise precision/recall, confusion matrix, and crossing-count error on the held-out set.
5. Export the validated custom model to ONNX and verify the browser's post-processing supports its class list before updating production. Cloudflare Pages has a per-file size limit, so check the exported model size.

**Important:** The V9 browser still loads the original COCO model (`yolo26n.onnx`). Training a custom model is an additional project step, not automatically performed by this notebook.

In [ ]:
# Optional training example. Does NOT run without your own dataset YAML and annotations.
# The labels, class order and deployment preprocessing MUST match your trained model.
from pathlib import Path
custom_dataset_yaml = Path("samples/custom-vehicle-dataset.yaml")
if custom_dataset_yaml.exists():
    print("Custom training dataset found. Train explicitly after reviewing its labels and train/val split.")
    print("Example: YOLO('yolo26n.pt').train(data=str(custom_dataset_yaml), epochs=50, imgsz=640)")
else:
    print("Custom dataset not provided; pretrained COCO-only model remains in use.")
